# Research Paper Answer Bot

## 1. Import Libraries

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

print("Libraries Imported Successfully")

Libraries Imported Successfully


## 2. Load Environment

In [2]:
def find_project_root(start_path: Path) -> Path:
    """Return the repository root containing the application and requirements."""
    for candidate in (start_path, *start_path.parents):
        if (candidate / "app").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate

    raise RuntimeError(
        "Project root not found. Start Jupyter from the Research-Paper-Answer-Bot directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ENV_PATH = PROJECT_ROOT / ".env"

if not ENV_PATH.is_file():
    raise FileNotFoundError(f"Environment file not found: {ENV_PATH}")

load_dotenv(dotenv_path=ENV_PATH, override=False)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


### Section 2 Tests

In [3]:
assert PROJECT_ROOT.is_dir(), "PROJECT_ROOT must be an existing directory."
assert (PROJECT_ROOT / "app").is_dir(), "PROJECT_ROOT must contain the app directory."
assert (PROJECT_ROOT / "requirements.txt").is_file(), "PROJECT_ROOT must contain requirements.txt."
print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")

PROJECT_ROOT verified: C:\Users\Alfin\OneDrive\Desktop\Research-Paper-Answer-Bot


In [4]:
assert ENV_PATH.is_file(), f".env file not found: {ENV_PATH}"
print(".env loading verified")

.env loading verified


In [5]:
if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY is missing. Add GOOGLE_API_KEY=your_key to the project .env file."
    )

print("GOOGLE_API_KEY verified")
print("Environment Loaded Successfully")

GOOGLE_API_KEY verified
Environment Loaded Successfully


## 3. Load Research Papers

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_documents(papers_directory: Path) -> list:
    """Load all readable PDFs below ``papers_directory``."""
    if not papers_directory.is_dir():
        raise FileNotFoundError(f"Papers directory not found: {papers_directory}")

    pdf_paths = sorted(
        path for path in papers_directory.rglob("*")
        if path.is_file() and path.suffix.lower() == ".pdf"
    )
    print(f"Total PDFs found: {len(pdf_paths)}")

    if not pdf_paths:
        raise FileNotFoundError(
            f"No PDF files were found in {papers_directory}. Add PDFs and run this cell again."
        )

    documents = []
    skipped_pdfs = []
    for pdf_path in pdf_paths:
        try:
            documents.extend(PyPDFLoader(str(pdf_path)).load())
        except Exception as error:
            skipped_pdfs.append(pdf_path)
            print(f"Skipping unreadable PDF: {pdf_path.name} ({error})")

    loaded_pdf_count = len(pdf_paths) - len(skipped_pdfs)
    print(f"Total PDFs successfully loaded: {loaded_pdf_count}")
    print(f"Skipped PDFs: {len(skipped_pdfs)}")
    return documents


PAPERS_DIRECTORY = PROJECT_ROOT / "app" / "data" / "papers"
documents = load_documents(PAPERS_DIRECTORY)
print("Section 3 verified")

Total PDFs found: 5
Total PDFs successfully loaded: 5
Skipped PDFs: 0
Section 3 verified


## 4. Read Research Papers

In [7]:
if not documents:
    raise RuntimeError(
        "No document pages were loaded. Check the PDFs in app/data/papers and run Section 3 again."
    )

first_document = documents[0]
display(first_document.metadata)
display(first_document.page_content[:1000])
print(f"Total loaded documents: {len(documents)}")
print("Section 4 verified")

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'C:\\Users\\Alfin\\OneDrive\\Desktop\\Research-Paper-Answer-Bot\\app\\data\\papers\\Attention.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurre

Total loaded documents: 168
Section 4 verified


## 5. Split Documents

In [8]:
if not documents:
    raise RuntimeError("No documents are available to split. Run Section 3 first.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)

if not chunks:
    raise RuntimeError("No chunks were created from the loaded documents.")

average_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)
print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk size: {average_chunk_size:.2f} characters")
display(chunks[0])
print("Section 5 verified")

Total chunks created: 722
Average chunk size: 873.21 characters


Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\Alfin\\OneDrive\\Desktop\\Research-Paper-Answer-Bot\\app\\data\\papers\\Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\n

Section 5 verified


## 6. Create Gemini Embedding Model

In [9]:
import time

from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY is missing. Add it to the project .env file before creating embeddings."
    )

try:
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-2",
        google_api_key=GOOGLE_API_KEY,
    )
    test_embedding = embeddings.embed_query("Hello World")
    if not test_embedding:
        raise ValueError("The embedding model returned an empty vector.")
except Exception as error:
    raise RuntimeError(
        "Gemini embedding test failed. Verify GOOGLE_API_KEY, model access, and network connectivity."
    ) from error

print(f"Gemini embedding test passed ({len(test_embedding)} dimensions).")
print("Section 6 verified")

Gemini embedding test passed (3072 dimensions).
Section 6 verified


## 7. Create Chroma Vector Database

In [10]:
import hashlib


VECTOR_DB_PATH = PROJECT_ROOT / "vector_db"
COLLECTION_NAME = "research_papers"
BATCH_SIZE = 2
MAX_RATE_LIMIT_RETRIES = 10


def is_rate_limit_error(error: Exception) -> bool:
    """Return whether an embedding request failed because of rate limiting."""
    status_code = getattr(error, "code", None)
    if callable(status_code):
        status_code = status_code()

    error_text = str(error).lower()
    return status_code == 429 or any(
        marker in error_text
        for marker in ("429", "resourceexhausted", "rate limit", "rate_limit")
    )


def chunk_id(chunk) -> str:
    """Create a stable ID so reruns can identify an already indexed chunk."""
    return hashlib.sha256(chunk.page_content.encode("utf-8")).hexdigest()


def persist_after_batch(vector_store: Chroma) -> None:
    """Persist a successful write for both current and older Chroma clients."""
    persist = getattr(vector_store._client, "persist", None)
    if callable(persist):
        persist()
    # Chroma PersistentClient (used above) persists each mutation automatically.


def add_batch_with_retries(
    vector_store: Chroma, batch: list, batch_ids: list[str], batch_number: int
) -> None:
    """Add one batch, retrying Gemini rate-limit responses."""
    for attempt in range(MAX_RATE_LIMIT_RETRIES + 1):
        try:
            vector_store.add_documents(batch, ids=batch_ids)
            persist_after_batch(vector_store)
            return
        except Exception as error:
            if not is_rate_limit_error(error):
                raise
            if attempt == MAX_RATE_LIMIT_RETRIES:
                raise RuntimeError(
                    f"Batch {batch_number} exceeded the Gemini rate-limit retry limit."
                ) from error

            delay_seconds = 60
            print(
                f"Rate limit reached for batch {batch_number}; retrying in {delay_seconds} seconds."
            )
            time.sleep(delay_seconds)


database_exists = VECTOR_DB_PATH.is_dir() and any(VECTOR_DB_PATH.iterdir())
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(VECTOR_DB_PATH),
)

if not chunks:
    raise RuntimeError("No chunks are available to index. Run Section 5 first.")

if database_exists:
    print("Existing vector database loaded; checking for chunks that still need indexing.")

stored_documents = vector_store.get(include=["documents"]).get("documents", [])
stored_chunk_ids = {
    hashlib.sha256(document.encode("utf-8")).hexdigest()
    for document in stored_documents
    if document is not None
}
pending_chunks = [chunk for chunk in chunks if chunk_id(chunk) not in stored_chunk_ids]
indexed_chunk_count = vector_store._collection.count()
total_batches = (len(pending_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

if not pending_chunks:
    print("All chunks are already indexed; no embeddings were recreated.")
    print("Current batch: 0")
    print("Total batches: 0")
    print(f"Indexed chunk count: {indexed_chunk_count}")
    print("Estimated remaining chunks: 0")
else:
    for batch_number, start_index in enumerate(
        range(0, len(pending_chunks), BATCH_SIZE), start=1
    ):
        batch = pending_chunks[start_index : start_index + BATCH_SIZE]
        batch_ids = [chunk_id(chunk) for chunk in batch]
        print(f"Current batch: {batch_number}")
        print(f"Total batches: {total_batches}")
        add_batch_with_retries(vector_store, batch, batch_ids, batch_number)

        indexed_chunk_count += len(batch)
        remaining_chunks = len(pending_chunks) - start_index - len(batch)
        print(f"Indexed chunk count: {indexed_chunk_count}")
        print(f"Estimated remaining chunks: {remaining_chunks}")
        time.sleep(5)

print(f"Database path: {VECTOR_DB_PATH}")
print("Section 7 verified")
print("Vector Database Created Successfully")

Existing vector database loaded; checking for chunks that still need indexing.
All chunks are already indexed; no embeddings were recreated.
Current batch: 0
Total batches: 0
Indexed chunk count: 722
Estimated remaining chunks: 0
Database path: C:\Users\Alfin\OneDrive\Desktop\Research-Paper-Answer-Bot\vector_db
Section 7 verified
Vector Database Created Successfully


## 8. Create Chroma Retriever

In [11]:
if "vector_store" not in globals():
    raise RuntimeError("Vector database is unavailable. Run Section 7 first.")

if vector_store._collection.count() == 0:
    raise RuntimeError("The vector database contains no chunks. Complete Section 7 before creating a retriever.")

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)
print("Chroma similarity retriever created with k=5.")
print("Section 8 verified")

Chroma similarity retriever created with k=5.
Section 8 verified


## 9. Create Gemini RetrievalQA Chain

In [12]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


try:
    ChatGoogleGenerativeAI
except NameError:
    from langchain_google_genai import ChatGoogleGenerativeAI


if not GOOGLE_API_KEY:
    raise EnvironmentError("GOOGLE_API_KEY is required to create the Gemini chat model.")

if "retriever" not in globals():
    raise RuntimeError("Retriever is unavailable. Run Section 8 first.")

try:
    chat_model = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0,
    )
except Exception as error:
    raise RuntimeError(
        "Gemini chat model initialization failed. Verify GOOGLE_API_KEY and model access."
    ) from error

if chat_model is None:
    raise RuntimeError("Gemini chat model initialization returned no model.")

try:
    qa_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Answer only from the supplied research-paper context. "
                "If the context does not contain the answer, say so clearly.\n\n"
                "<context>\n{context}\n</context>",
            ),
            ("human", "{input}"),
        ]
    )
    document_chain = create_stuff_documents_chain(chat_model, qa_prompt)
    retrieval_qa_chain = create_retrieval_chain(retriever, document_chain)
except Exception as error:
    raise RuntimeError(
        "Could not create the Gemini RetrievalQA chain. Verify the API key, model access, and Sections 8 dependencies."
    ) from error

print("Gemini RetrievalQA chain created successfully.")
print("Section 9 verified")

Gemini RetrievalQA chain created successfully.
Section 9 verified


## 10. Ask Research Questions

In [13]:
def ask_question(question: str) -> dict:
    """Run a question through the retrieval chain and display its result."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty research question.")

    started_at = time.perf_counter()
    try:
        result = retrieval_qa_chain.invoke({"input": question.strip()})
    except Exception as error:
        raise RuntimeError(
            "The question could not be answered. Check Gemini API access and try again."
        ) from error

    response_time = time.perf_counter() - started_at
    source_documents = result.get("context", [])
    answer = result.get("answer", "No answer was returned.")

    display({"Answer": answer})
    display({"Source documents": source_documents})
    display({"Response time": f"{response_time:.2f} seconds"})
    display({"Retrieved document count": len(source_documents)})

    return {
        "answer": answer,
        "source_documents": source_documents,
        "response_time": response_time,
    }


while True:
    user_question = input("Ask a research-paper question (or type 'exit'): " ).strip()
    if user_question.lower() in {"exit", "quit"}:
        print("Interactive question session ended.")
        break

    try:
        ask_question(user_question)
    except (RuntimeError, ValueError) as error:
        print(f"Error: {error}")

{'Answer': 'Based on the provided context, Retrieval-Augmented Generation (RAG) is a general-purpose fine-tuning approach and model class for language generation. It combines two types of memory:\n\n1. **Parametric memory:** A pre-trained sequence-to-sequence (seq2seq) model (such as a transformer like BART).\n2. **Non-parametric memory:** An external, explicit memory consisting of a dense vector index of Wikipedia, which is accessed using a pre-trained neural retriever (such as the Dense Passage Retriever). \n\nThese components are combined in a probabilistic model trained end-to-end, allowing the model to retrieve latent documents conditioned on the input and use them to generate the final output.'}

{'Source documents': [Document(id='0cec141aabea0ed1a2535b4575f9f8fcc2c269208e1762c0dcf8cd3ca5130064', metadata={'trapped': '/False', 'creator': 'LaTeX with hyperref', 'page_label': '1', 'total_pages': 19, 'source': 'C:\\Users\\Alfin\\OneDrive\\Desktop\\Research-Paper-Answer-Bot\\app\\data\\papers\\Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf', 'title': '', 'moddate': '2021-04-13T00:48:38+00:00', 'author': '', 'producer': 'pdfTeX-1.40.21', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'creationdate': '2021-04-13T00:48:38+00:00', 'keywords': '', 'page': 0}, page_content='decisions and updating their world knowledge remain open research problems. Pre-\ntrained models with a differentiable access mechanism to explicit non-parametric\nmemory have so far been only investigated for extractive downstream tasks. We\nexplore a general-purpose ﬁne-tuning recipe for retrieval-augmented generation\n

{'Response time': '12.29 seconds'}

{'Retrieved document count': 5}

Interactive question session ended.


### Final Validation

In [14]:
assert retriever.search_type == "similarity", "Retriever must use similarity search."
assert retriever.search_kwargs.get("k") == 5, "Retriever k must be 5."
assert retrieval_qa_chain is not None, "RetrievalQA chain was not created."
assert callable(ask_question), "Interactive question function is unavailable."
print("Research Paper Answer Bot Ready")

Research Paper Answer Bot Ready
